# Fabric Anomaly Detection with AnomalyCLIP

This notebook trains **AnomalyCLIP** (Zhou et al., ICLR 2024) on your fabric
defect dataset, and is set up to run directly as a **Kaggle Notebook**
(dataset attached via `/kaggle/input/...`, outputs written to
`/kaggle/working/...`), following this pipeline:

```
Input Image -> CLIP Image Encoder -> Multi-scale Patch Features
            -> Compare with Text Prompts -> Similarity Score Per Patch
            -> Pixel Anomaly Map -> Threshold / Heatmap
            -> Defect Localization Result
```

**What AnomalyCLIP actually does under the hood** (so the diagram makes sense):
- CLIP's image encoder is run with a modified **V-V (value-value) self-attention**
  in the last few transformer blocks, which makes the patch tokens spatially
  local and suitable for *segmentation* rather than just global classification.
- Instead of hand-written prompts like `"a photo of a defect"`, AnomalyClip
  **learns** two prompt embeddings end-to-end: one that represents "normal"
  and one that represents "abnormal" (this is called *object-agnostic prompt
  learning* — it does not need the class name "fabric" at all).
- Patch tokens from **multiple transformer layers** (multi-scale) are each
  projected into CLIP's text-embedding space through small linear adapters.
- Each patch embedding is compared (cosine similarity) against the learned
  normal/abnormal text embeddings -> a per-patch anomaly score.
- Scores are reshaped into a low-res grid, upsampled to the original image
  size (bilinear interpolation) -> the pixel-level anomaly heatmap.
- A single global anomaly score for the whole image is the max (or mean-topk)
  of that map.

**Important scope note:** AnomalyCLIP is designed for *zero/few-shot* anomaly
detection — it is pretrained on auxiliary defect datasets (MVTec-AD, VisA)
and then generalizes to new objects with little/no fine-tuning. For fabric,
you have two realistic options, both supported below:

1. **Zero-shot / direct inference** — use the AnomalyCLIP checkpoint that the
   authors already trained on MVTec-AD+VisA and just run it on your fabric
   images (no training needed, works surprisingly well).
2. **Fine-tune** — continue training the prompts + adapters on your fabric
   dataset for better accuracy (recommended if you have a labelled few-shot
   set with masks or even just good/defect labels).

This notebook does **both**: it fine-tunes on your data, and shows how to
skip straight to inference if you'd rather not fine-tune.

---

### Before you run this
1. On the right-hand side panel of the Kaggle Notebook editor, set
   **Settings -> Accelerator -> GPU T4 x2** (or any available GPU) and turn
   **Internet: On** (needed to `git clone` the AnomalyCLIP repo and download
   CLIP weights).
2. Click **Add Input** (top right) and attach your fabric dataset. Once
   attached it will be mounted read-only at `/kaggle/input/<your-dataset-slug>/`.
3. Update `RAW_DATA_DIR` in step 2 below to match your dataset's exact path
   under `/kaggle/input/` (check the right-hand "Input" panel for the exact
   folder name — it's usually the dataset slug).
4. Know your dataset's folder structure — run the "Explore dataset" cell and
   adjust the `collect_samples()` function in the dataset config cell to match.


## 1. Environment check

In [ ]:
# Confirms a GPU is attached and shows which one.
# AnomalyCLIP's backbone is a CLIP ViT (e.g. ViT-L/14@336px) -> training
# and even inference is impractically slow on CPU.
import torch                                   # PyTorch, the deep learning framework everything else is built on
print("CUDA available:", torch.cuda.is_available())   # True/False - must be True
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))   # prints e.g. "Tesla T4"


## 2. Clone the official AnomalyCLIP repo and install dependencies

We use the authors' own implementation (`github.com/zqhang/AnomalyCLIP`) so
the V-V attention patch, prompt-learning module, and multi-scale adapter are
guaranteed to match the paper exactly — reimplementing these by hand is the
most common source of subtle bugs.


In [ ]:
import os
os.chdir("/kaggle/working")                # /kaggle/working is writable; /kaggle/input is read-only

# Remove any previous partial clone, then clone the official repo fresh.
!rm -rf AnomalyCLIP
!git clone https://github.com/zqhang/AnomalyCLIP.git

%cd /kaggle/working/AnomalyCLIP
# Editable install of the repo's own CLIP fork (it patches CLIP's attention
# layers, so you must use THIS version, not pip's vanilla `clip` package).
!pip install -q -e .

# Extra libraries the training/eval scripts use: image IO, metrics, progress bars.
!pip install -q opencv-python scikit-image scikit-learn tqdm ftfy regex


## 3. Point the notebook at your attached dataset

Because this is a Kaggle Notebook, there is no download step — your dataset
is already mounted read-only under `/kaggle/input/` once you've attached it
via **Add Input**. We just need the exact path.


In [ ]:
import os

# List everything currently attached under /kaggle/input so you can copy the
# exact folder name into RAW_DATA_DIR below (Kaggle mounts each attached
# dataset as /kaggle/input/<dataset-slug>/).
print("Attached datasets:")
for name in os.listdir("/kaggle/input"):
    print(" -", name)

# ---- EDIT THIS: replace with the exact folder name printed above.
RAW_DATA_DIR = "/kaggle/input/your-fabric-dataset-slug"

assert os.path.exists(RAW_DATA_DIR), f"Path not found: {RAW_DATA_DIR} -- check the printed list above and fix RAW_DATA_DIR"
print("\nDataset will be read from:", RAW_DATA_DIR)


## 4. Explore the dataset structure

Run this and look at the printed tree **before** touching the config cell
below — fabric datasets on Kaggle come in very different layouts (some have
`good/` `defect/` folders, some have a CSV of labels, some have
per-defect-type subfolders like `hole/`, `stain/`, `broken_thread/`).


In [ ]:
import os

def print_tree(root, max_depth=3, max_items=15):
    # Walks the folder structure and prints a shallow tree so you can see
    # how images/labels/masks are organized, without dumping thousands of filenames.
    root = os.path.abspath(root)
    for cur_root, dirs, files in os.walk(root):
        depth = cur_root[len(root):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []   # stop descending further
            continue
        indent = "  " * depth
        print(f"{indent}{os.path.basename(cur_root) or cur_root}/")
        sub_indent = "  " * (depth + 1)
        for f in files[:max_items]:
            print(f"{sub_indent}{f}")
        if len(files) > max_items:
            print(f"{sub_indent}... ({len(files) - max_items} more files)")

print_tree(RAW_DATA_DIR)


## 5. Dataset config — matched to your layout

Your dataset (confirmed from the explore step) already looks like this:

```
dataset/
  train/
    good/
  test/
    good/
    holes/
    lines/
    stains/
```

This is basically already MVTec-AD-style, just with three separate defect
types instead of one `defect/` folder — so instead of a 90/10 split, we use
your existing `train/` and `test/` folders directly, and keep `holes`,
`lines`, `stains` as three distinct defect types (so evaluation later can
report accuracy per defect type, not just overall).

**Set `RAW_DATA_DIR` below to the folder that directly contains `train/` and
`test/`** — i.e. the `dataset` folder from your screenshot, for example:
`/kaggle/input/dataset-for-fabric-anomaly-detection-for/EfficientNetB0/dataset`
(copy the exact path from your Step 4 tree output).


In [ ]:
import shutil
from pathlib import Path

# ---- EDIT THIS if needed: must point at the folder that directly contains train/ and test/
RAW_DATA_DIR = Path(RAW_DATA_DIR)  # reuse the path you set in Step 3
if not (RAW_DATA_DIR / "train" / "good").exists():
    # Common case: RAW_DATA_DIR was set one level too high (e.g. .../EfficientNetB0
    # instead of .../EfficientNetB0/dataset). Try to auto-descend into "dataset".
    candidate = RAW_DATA_DIR / "dataset"
    if (candidate / "train" / "good").exists():
        RAW_DATA_DIR = candidate
        print("Auto-adjusted RAW_DATA_DIR ->", RAW_DATA_DIR)

assert (RAW_DATA_DIR / "train" / "good").exists(), \
    f"Could not find train/good under {RAW_DATA_DIR} -- fix RAW_DATA_DIR in Step 3"

CLASS_NAME = "fabric"                                  # AnomalyCLIP treats each object/texture type as a "class"
DATASET_ROOT = Path("/kaggle/working/mvtec_format")    # /kaggle/working is the writable output area
DEFECT_TYPES = ["holes", "lines", "stains"]             # the three defect subfolders under test/
HAS_MASKS = False                                       # no ground_truth/mask folders in this dataset

# --- Build MVTec-style folder tree -----------------------------------------
train_good_dir = DATASET_ROOT / CLASS_NAME / "train" / "good"
test_good_dir  = DATASET_ROOT / CLASS_NAME / "test"  / "good"
test_defect_dirs = {d: DATASET_ROOT / CLASS_NAME / "test" / d for d in DEFECT_TYPES}

for d in [train_good_dir, test_good_dir, *test_defect_dirs.values()]:
    d.mkdir(parents=True, exist_ok=True)                # create folders, no error if they exist

# ---- copy train/good ----
train_good_src = sorted((RAW_DATA_DIR / "train" / "good").glob("*.*"))
for f in train_good_src:
    shutil.copy(f, train_good_dir / f.name)             # copy, keep original filename

# ---- copy test/good ----
test_good_src = sorted((RAW_DATA_DIR / "test" / "good").glob("*.*"))
for f in test_good_src:
    shutil.copy(f, test_good_dir / f.name)

# ---- copy each defect type under test/ ----
defect_counts = {}
for dtype in DEFECT_TYPES:
    src_files = sorted((RAW_DATA_DIR / "test" / dtype).glob("*.*"))
    for f in src_files:
        shutil.copy(f, test_defect_dirs[dtype] / f.name)
    defect_counts[dtype] = len(src_files)

print("Train (good):", len(list(train_good_dir.glob('*'))))
print("Test (good):", len(list(test_good_dir.glob('*'))))
for dtype, count in defect_counts.items():
    print(f"Test ({dtype}):", count)


## 6. Generate the `meta.json` index file

AnomalyCLIP's data loader reads a `meta.json` describing every image's path,
label (0=normal, 1=anomaly), and mask path (if available). This cell builds
it automatically from the folder structure created above, keeping `holes`,
`lines`, and `stains` as separate defect types so you get per-type metrics
later, not just an overall good-vs-defect number.


In [ ]:
import json

meta = {CLASS_NAME: {"train": [], "test": []}}

# ---- training split: only normal ("good") images ----
for f in sorted(train_good_dir.glob("*")):
    meta[CLASS_NAME]["train"].append({
        "img_path": f"{CLASS_NAME}/train/good/{f.name}",
        "mask_path": "",                # no mask needed for normal training images
        "cls_name": CLASS_NAME,
        "specie_name": "good",
        "anomaly": 0                    # 0 = normal
    })

# ---- test split: normal images ----
for f in sorted(test_good_dir.glob("*")):
    meta[CLASS_NAME]["test"].append({
        "img_path": f"{CLASS_NAME}/test/good/{f.name}",
        "mask_path": "",
        "cls_name": CLASS_NAME,
        "specie_name": "good",
        "anomaly": 0
    })

# ---- test split: each defect type (holes, lines, stains) ----
for dtype, dpath in test_defect_dirs.items():
    for f in sorted(dpath.glob("*")):
        mask_rel = f"{CLASS_NAME}/ground_truth/{dtype}/{f.stem}_mask.png" if HAS_MASKS else ""
        meta[CLASS_NAME]["test"].append({
            "img_path": f"{CLASS_NAME}/test/{dtype}/{f.name}",
            "mask_path": mask_rel,      # empty string since this dataset has no pixel-level masks
            "cls_name": CLASS_NAME,
            "specie_name": dtype,       # keeps "holes"/"lines"/"stains" distinct for per-type metrics
            "anomaly": 1                # 1 = anomalous
        })

meta_path = DATASET_ROOT / "meta.json"
with open(meta_path, "w") as fp:
    json.dump(meta, fp, indent=2)       # write the index as pretty-printed JSON

print("Wrote", meta_path)
print("Train images:", len(meta[CLASS_NAME]["train"]))
print("Test images:", len(meta[CLASS_NAME]["test"]))


## 7. Download the pretrained CLIP backbone weights

AnomalyCLIP fine-tunes prompts/adapters on top of a **frozen, pretrained
CLIP** (ViT-L/14 @ 336px by default). This downloads those base weights
(not the AnomalyCLIP-specific checkpoint yet).


In [ ]:
# open_clip provides the pretrained CLIP weights AnomalyCLIP's backbone expects.
!pip install -q open_clip_torch

import open_clip
# Downloads and caches the ViT-L-14-336 weights pretrained on OpenAI's data.
# This just warms the local cache so the training script below doesn't stall on first download.
_ = open_clip.create_model_and_transforms("ViT-L-14-336", pretrained="openai")
print("CLIP backbone weights cached.")


## 8. Fine-tune AnomalyCLIP on your fabric data

This calls the repo's own `train.py`. Only the **learnable prompts and
lightweight adapters** are trained — the CLIP backbone stays frozen, so this
is fast (a few minutes to ~1 hour on a single T4, depending on dataset size)
and needs little data.

Flag reference:
- `--train_data_path` : the MVTec-format root folder we just built
- `--dataset` : tells the script which meta.json parsing rules to use (`mvtec` layout works for our custom folder)
- `--save_path` : where trained checkpoints (prompt + adapter weights) are written
- `--features_list` : which CLIP transformer layers to pull multi-scale patch features from
- `--image_size` : input resolution fed to CLIP (336 matches the ViT-L-14-336 backbone)
- `--epoch` : training epochs — AnomalyCLIP converges fast since so few parameters are trained
- `--batch_size` : images per training step
- `--print_freq` : how often to log loss to the console


In [ ]:
!python train.py \
    --train_data_path {str(DATASET_ROOT)} \
    --dataset mvtec \
    --save_path ./checkpoints/fabric_anomalyclip \
    --features_list 6 12 18 24 \
    --image_size 336 \
    --batch_size 8 \
    --epoch 15 \
    --print_freq 10


## 9. Run inference / evaluation on the test split

`test.py` loads the checkpoint saved above, runs every test image through
the pipeline in the diagram (encode -> compare to learned prompts -> patch
similarity -> upsample to full-res heatmap), and reports metrics
(image-level AUROC always; pixel-level AUROC/AUPRO too if you supplied masks).


In [ ]:
!python test.py \
    --data_path {str(DATASET_ROOT)} \
    --dataset mvtec \
    --checkpoint_path ./checkpoints/fabric_anomalyclip/epoch_15.pth \
    --features_list 6 12 18 24 \
    --image_size 336 \
    --save_path ./results/fabric_anomalyclip


## 10. Visualize a defect localization result

Overlays the predicted pixel anomaly map on the original image, exactly as
the last step of your diagram ("Defect Localization Result").


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show_anomaly_map(image_path, anomaly_map_path, threshold=0.5):
    # Load the original fabric image (BGR -> RGB for correct display colors).
    img = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)

    # Load the model's predicted per-pixel anomaly score map (produced by test.py,
    # saved as a single-channel image or .npy array with values in [0, 1]).
    amap = np.load(anomaly_map_path) if str(anomaly_map_path).endswith(".npy") \
        else cv2.imread(str(anomaly_map_path), cv2.IMREAD_GRAYSCALE) / 255.0

    # Binary mask of "defect" pixels: anywhere the anomaly score exceeds threshold.
    binary_mask = (amap > threshold).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img);                       axes[0].set_title("Input Image");           axes[0].axis("off")
    axes[1].imshow(amap, cmap="jet");          axes[1].set_title("Anomaly Heatmap");        axes[1].axis("off")
    axes[2].imshow(img)
    axes[2].imshow(amap, cmap="jet", alpha=0.5)   # heatmap overlaid on the original image
    axes[2].set_title(f"Localization (threshold={threshold})")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()

# Example usage — update these paths to an actual test image and its saved anomaly map:
# show_anomaly_map(
#     image_path="mvtec_format/fabric/test/defect/0001.jpg",
#     anomaly_map_path="results/fabric_anomalyclip/fabric/defect/0001.npy",
#     threshold=0.5
# )


## 11. (Optional) Zero-shot inference — skip fine-tuning entirely

If you'd rather just test AnomalyCLIP's official pretrained checkpoint
(trained by the authors on MVTec-AD + VisA) directly on your fabric images
with no training at all, download their released weights and point
`--checkpoint_path` at that file instead of your fine-tuned one in step 9.
Zero-shot performance on textures like fabric is often already strong,
so it's worth trying this first as a baseline before investing time in
fine-tuning.

See the repo's README for the pretrained checkpoint download link:
https://github.com/zqhang/AnomalyCLIP


## Tips & troubleshooting

- **Out of memory**: lower `--batch_size` (try 4 or 2) or drop `--image_size` to 240.
- **No masks available**: pixel-level AUROC/AUPRO just won't be computed —
  image-level (good vs. defect) AUROC still works fine with `mask_path: ""`.
- **Multiple defect types**: already handled — `holes`, `lines`, and
  `stains` are kept as separate `specie_name` values in `meta.json`, so
  `test.py`'s evaluation report will break down accuracy per defect type
  as well as overall.
- **Very small dataset**: AnomalyCLIP only trains a small number of prompt
  and adapter parameters, so even ~50-100 "good" images can work — the
  bottleneck is usually the size of your labelled test set for evaluation,
  not the training set.
